# RG3 gas: dimensionless process coupling ensembles
18 predeclared derived features in three six-feature groups. RF + ExtraTrees mean/max ensembles.
No timestamp predictor, no LH merging, no test-label threshold tuning. Peak pressure times fill duration is only a proxy, not measured energy. Cycle fractions are not true residence time.
Same observation-block evaluation as baseline; exploratory model selection requires new-block validation.

In [ ]:
from pathlib import Path
import json
import numpy as np,pandas as pd
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
nb=json.loads((root/'notebooks/22_rg3_nested_signal_ensemble.ipynb').read_text(encoding='utf-8'))
exec(''.join(nb['cells'][1]['source']).split('scores=[];thresholds=[];foldlogs=[]')[0])
def ratio(a,b):
    return a/np.where(np.abs(b)>1e-6,b,np.nan)
F=pd.DataFrame(index=d.index)
F['switch_pressure_fraction']=ratio(d.Max_Switch_Over_Pressure,d.Max_Injection_Pressure)
F['back_pressure_peak_fraction']=ratio(d.Max_Back_Pressure,d.Average_Back_Pressure)
F['pressure_per_speed']=ratio(d.Max_Injection_Pressure,d.Max_Injection_Speed)
F['fill_fraction']=ratio(d.Filling_Time,d.Injection_Time)
F['cushion_fraction']=ratio(d.Cushion_Position,d.Plasticizing_Position)
F['pressure_time_proxy']=d.Max_Injection_Pressure*d.Filling_Time
F['plasticizing_cycle_fraction']=ratio(d.Plasticizing_Time,d.Cycle_Time)
F['cycle_unaccounted_time']=d.Cycle_Time-d.Injection_Time-d.Plasticizing_Time-d.Clamp_Close_Time
F['barrel_rear_front_gap']=d.Barrel_Temperature_2-d.Barrel_Temperature_5
F['hopper_barrel_gap']=d.Hopper_Temperature-d.Barrel_Temperature_5
F['mold_side_gap']=d.Mold_Temperature_3-d.Mold_Temperature_4
F['thermal_hopper_barrel_shake_ratio']=ratio(d.Hopper_Temperature__past10_std,d.Barrel_Temperature_5__past10_std)
# Departures relative to each sensor's own preceding 10-shot variability.
for sensor in ['Cushion_Position','Max_Injection_Pressure','Average_Back_Pressure','Barrel_Temperature_2']:
    F[sensor+'_relative_departure']=ratio(d[sensor+'__dev10'],d[sensor+'__past10_std'])
F['pressure_cushion_departure_product']=F.Max_Injection_Pressure_relative_departure*F.Cushion_Position_relative_departure
F['pressure_back_departure_difference']=F.Max_Injection_Pressure_relative_departure-F.Average_Back_Pressure_relative_departure
F=F.replace([np.inf,-np.inf],np.nan)
views={'transmission6':F.columns[:6].tolist(),'recovery_thermal6':F.columns[6:12].tolist(),'coupling6':F.columns[12:].tolist(),'combined18':F.columns.tolist()}
memo={}
def predict(ref,q):
    key=(tuple(ref),tuple(q))
    if key in memo:return memo[key]
    assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
    out={}
    for name,cols in views.items():
        if len(np.unique(y[ref]))<2:
            out[name+'_mean']=out[name+'_max']=np.zeros(len(q));continue
        s=[]
        for cls in [RandomForestClassifier,ExtraTreesClassifier]:
            m=make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),cls(n_estimators=100,max_depth=5,min_samples_leaf=5,class_weight='balanced',random_state=42,n_jobs=2))
            s.append(m.fit(F.iloc[ref][cols],y[ref]).predict_proba(F.iloc[q][cols])[:,1])
        out[name+'_mean']=np.mean(s,axis=0)
        out[name+'_max']=np.max(s,axis=0)
    memo[key]=out
    return out
records=[];audit=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,b in enumerate(blocks):
        q=np.flatnonzero(groups.eq(b))
        ref=np.flatnonzero(~groups.eq(b)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
        if len(np.unique(y[ref]))<2:continue
        outer=predict(ref,q);inner={n:np.full(len(ref),np.nan) for n in outer}
        for ib in groups.iloc[ref].unique():
            mask=groups.iloc[ref].eq(ib).to_numpy()
            if not (~mask).any():continue
            p=predict(ref[~mask],ref[mask])
            for n in inner:inner[n][mask]=p[n]
        for name,score in outer.items():
            cal=inner[name];valid=np.isfinite(cal);norm=cal[valid&(y[ref]==0)]
            cut=float(np.quantile(norm,.99,method='higher')) if len(norm) else 1.
            chosen=cut;best=-1.
            for t in np.unique(np.r_[cal[valid],1.]):
                pr=cal[valid]>t;yy=y[ref][valid]
                fp=int(np.sum(pr&(yy==0)));tp=int(np.sum(pr&(yy==1)));fn=int(np.sum(~pr&(yy==1)))
                if fp>int(np.floor(.02*np.sum(yy==0))):continue
                f=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
                if f>best or (f==best and t>chosen):best=f;chosen=float(t)
            for rule,t in [('normal99',cut),('inner_f1_fp2pct',chosen)]:
                for k,i in enumerate(q):records.append(dict(scheme=scheme,model=name,rule=rule,TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(score[k]),threshold=t,prediction=int(score[k]>t)))
                audit.append(dict(scheme=scheme,block=b,model=name,rule=rule,train=len(ref),train_gas=int(y[ref].sum()),threshold=t,inner_best_F1=best))
        print('Finished',scheme,b,flush=True)
pp=pd.DataFrame(records)
base=pd.read_csv(tables/'rg3_nested_ensemble_predictions.csv',parse_dates=['TimeStamp'])
base=base.loc[base.model.eq('max_union'),['scheme','TimeStamp','predicted_train_fpr1']]
rows=[]
for (scheme,name,rule),g in pp.groupby(['scheme','model','rule'],sort=False):
    z=g.merge(base,on=['scheme','TimeStamp'],validate='one_to_one')
    for combine in ['new_only','OR_RF','AND_RF']:
        pred=z.prediction.astype(bool)
        if combine=='OR_RF':pred=pred|z.predicted_train_fpr1
        if combine=='AND_RF':pred=pred&z.predicted_train_fpr1
        tn,fp,fn,tp=confusion_matrix(z.actual,pred,labels=[0,1]).ravel()
        rows.append(dict(scheme=scheme,model=name,rule=rule,combine=combine,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.,AP=average_precision_score(z.actual,z.score),shots=len(z)))
metrics=pd.DataFrame(rows)
F.assign(TimeStamp=d.TimeStamp).to_csv(tables/'rg3_dimensionless_coupling_features.csv',index=False,encoding='utf-8-sig')
pp.to_csv(tables/'rg3_dimensionless_coupling_predictions.csv',index=False,encoding='utf-8-sig')
metrics.to_csv(tables/'rg3_dimensionless_coupling_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(audit).to_csv(tables/'rg3_dimensionless_coupling_fold_audit.csv',index=False,encoding='utf-8-sig')
for scheme in metrics.scheme.unique():
    print('\n',scheme)
    print(metrics.loc[metrics.scheme.eq(scheme)].sort_values(['F1','FP'],ascending=[False,True]).head(12).to_string(index=False))


## Executed results
Whole evaluation: no candidate exceeds baseline F1=0.296.
Forward RF OR coupling6_max, normal99: TP3 FP11 FN10 TN416 F1=0.222. Whole: TP4 FP12 FN13 TN554 F1=0.242.
Forward-only additional gas: 2020-10-22 04:51:46. This is exploratory and not a validated replacement.
See reports/rg3_geometry_and_coupling_comparison.md and tables.